# Model explanation (SHAP)

Explains one trained TOK model with SHAP `DeepExplainer`. Set the station, W&B run ID and dates in the first cell.

In [ ]:
# Earlier copies of this notebook were run with these settings:
#   station 4230, run t3b6sfbc, test year 2023, date 2023-08-03
#   station 8500, run k7uwmqpa, test year 2023, date 2023-10-27
#   station 9015, run x3g63dys, test year 2023, date 2023-08-03, output hour index 15 instead of 17
# Those copies used the last 1000 training samples as SHAP background instead of 2000.
station_id = 4222
run_id = "iksnird6"  # checkpoint is read from modelsSweep/final_model_<run_id>.pth
test_year = 2022
explain_date = "2022-09-15"

In [ ]:
# Load the datasets
from TOK_dataset import TOKDataset

# Common parameters for datasets
common_params = {
    'meteo_past_hours': 48,
    'prediction_hours': 18,
    'river_history_hours': 12,
    'num_closest_meteo_stations': 1
}

# Training data: 2012 up to the test year
dataset_train = TOKDataset(
    station_id=station_id,  # Using station ID from gin config
    from_year=2012,
    up_to_year=test_year,
    generate_extra_data_points=True,
    **common_params
)

# Test data: the test year
dataset_test = TOKDataset(
    station_id=station_id,
    from_year=test_year,
    up_to_year=test_year + 1,
    generate_extra_data_points=False,
    predefined_scalers=dataset_train.get_scalers(),
    **common_params
)



In [ ]:
import torch
import gin
from TOK_train import Config

# Load the gin config for TOK
gin.parse_config_file("TOK_config.gin")

# Path to the model checkpoint
model_path = f"modelsSweep/final_model_{run_id}.pth"

# Load the checkpoint
checkpoint = torch.load(model_path, map_location='cpu')

# Import the TOK model class, which will now use the gin config
from TOK.TOK import TOK
model = TOK()

print("Model initialized. Dynamic layers will be created during first forward pass.")
print("Available checkpoint keys:", list(checkpoint.keys()))

# Note: We'll load the model parameters after we do a forward pass to initialize dynamic layers


In [ ]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(dataset_train, batch_size=256, shuffle=False)
# Get a sample batch from test dataset
sample_batch = next(iter(train_dataloader))

# Set model to evaluation mode
model.eval()

print("\n=== STEP 1: Forward pass to initialize dynamic layers ===")
# Do a forward pass to create the dynamic fusion layer
with torch.no_grad():
    _ = model(sample_batch)
    
print("Dynamic layers created! Model state after first forward pass:")
print(f"Fusion layer exists: {model.fusion_layer is not None}")
if model.fusion_layer is not None:
    print(f"Fusion layer structure: {model.fusion_layer}")

print("\n=== STEP 2: Load saved parameters ===")
# Now load the saved parameters - the dynamic layers should match
try:
    model.load_state_dict(checkpoint['model_state_dict'])
    print("✓ Model parameters loaded successfully!")
except Exception as e:
    print(f"⚠ Loading with strict=False due to: {e}")
    model.load_state_dict(checkpoint['model_state_dict'], strict=True)

print("\n=== STEP 3: Model ready for inference ===")
output_scaler = dataset_train.scalers['Q'] # sklearn scaler
model.output_scaler = output_scaler

# Make a proper forward pass with loaded parameters

test_dataloader = DataLoader(dataset_test, batch_size=256, shuffle=False)
train_dataloader = DataLoader(dataset_train, batch_size=256, shuffle=False)


import numpy as np
with torch.no_grad():
    all_outputs = []
    all_targets = []
    all_timestamps = []
    
    # Process all batches
    for batch in test_dataloader:
        output = model(batch)
        scaled_output = model.scaled_predictions(output)
        scaled_target = model.scaled_predictions(batch['target'])
        prediction_timestamps = batch['prediction_timestamp']
        
        all_outputs.append(scaled_output)
        all_targets.append(scaled_target)
        all_timestamps.append(prediction_timestamps)
    
    # Concatenate all batches
    scaled_output = np.concatenate(all_outputs, axis=0)
    scaled_target = np.concatenate(all_targets, axis=0)
    prediction_timestamps = np.concatenate(all_timestamps, axis=0)
    
    print("Scaled predictions shape:", scaled_output.shape)
    print("Prediction timestamps shape:", prediction_timestamps.shape)

print(f"\nLast model output shape: {output.shape}")
print("Model is now ready for inference!")

In [ ]:
shapes = {}
for key in sample_batch.keys():
    if hasattr(batch[key], 'shape'):
        shapes[key] = batch[key].shape
    else:
        shapes[key] = type(batch[key])
shapes

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt
import torch

# Create a wrapper class to handle dictionary inputs
class ModelWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.device = next(model.parameters()).device
        
    def forward(self, x):
        # Convert flattened input back to dictionary format
        batch_size = x.shape[0]
        feature_dict = {}
        start_idx = 0
        
        # Reconstruct dictionary with original shapes from shapes dictionary
        for key, shape in shapes.items():
            if key not in ['target', 'prediction_timestamp'] and isinstance(shape, tuple):
                feature_size = torch.prod(torch.tensor(shape[1:])).item()  # Get size of features excluding batch dim
                feature_dict[key] = x[:, start_idx:start_idx + feature_size].reshape(batch_size, *shape[1:])
                start_idx += feature_size
                
        # Get model output and ensure it has the right shape for SHAP
        output = self.model(feature_dict)[:,:,17]
        return output

# Prepare background data
def flatten_batch_features(batch, shapes):
    features = []
    for key, shape in shapes.items():
        if key not in ['target', 'prediction_timestamp'] and isinstance(shape, tuple):
            value = batch[key]
            if isinstance(value, torch.Tensor):
                features.append(value.reshape(value.shape[0], -1))
    return torch.cat(features, dim=1)

background_data = []
for batch in train_dataloader:
    background_data.append(flatten_batch_features(batch, shapes))
background_data = torch.cat(background_data, dim=0)

# Get test data for SHAP analysis
test_data = []
test_dates = []
raw_test_data = []
for batch in test_dataloader:
    # Flatten each sample in the batch into a single dictionary
    for i in range(len(batch['target'])):
        sample_dict = {k: v[i] if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        raw_test_data.append(sample_dict)
    test_data.append(flatten_batch_features(batch, shapes))
    test_dates.append(batch['prediction_timestamp'])
test_data = torch.cat(test_data, dim=0)
test_dates = torch.cat(test_dates, dim=0)

print("Test background data shape:", test_data.shape)

# Convert test dates to datetime objects
import pandas as pd
test_dates = [pd.to_datetime(date, unit='s') for date in test_dates]

# Find index of the sample to explain
target_date = pd.Timestamp(explain_date)
sample_idx = None
for i, date in enumerate(test_dates):
    if date.date() == target_date.date():
        sample_idx = i
        break

if sample_idx is not None:
    sample_to_explain = test_data[[sample_idx]]
    print(f"Found sample from {target_date}")
else:
    print(f"No sample found for {target_date}")
    sample_to_explain = None


print(raw_test_data[sample_idx]['target'], scaled_target[sample_idx])
# Create wrapped model and SHAP explainer with increased tolerance
wrapped_model = ModelWrapper(model)
explainer = shap.DeepExplainer(wrapped_model, background_data[-2000:])
shap_values = explainer.shap_values(sample_to_explain, check_additivity=False)  # Disable additivity check

def reconstruct_shap_values(shap_values, shapes):
    """
    Reconstruct SHAP values into a dictionary format matching the original feature shapes.
    
    Args:
        shap_values: SHAP values from the explainer
        shapes: Dictionary containing the original shapes of each feature
        
    Returns:
        Dictionary containing SHAP values reshaped to match original feature shapes
    """
    shap_dict = {}
    
    # Get the feature names and their corresponding indices
    feature_indices = {}
    start_idx = 0
    for key, shape in shapes.items():
        if key not in ['target', 'prediction_timestamp'] and isinstance(shape, tuple):
            feature_size = torch.prod(torch.tensor(shape[1:])).item()
            feature_indices[key] = (start_idx, start_idx + feature_size)
            start_idx += feature_size

    # Reconstruct SHAP values into dictionary format
    for key, (start, end) in feature_indices.items():
        # Get SHAP values for this feature
        feature_shap = shap_values[0, start:end]
        
        # Reshape back to original feature shape (excluding batch dimension)
        original_shape = shapes[key][1:]  # Get shape without batch dimension
        feature_shap = feature_shap.reshape(original_shape)
        
        # Store in dictionary
        shap_dict[key] = feature_shap

    # Print shapes of SHAP values for each feature
    #for key, value in shap_dict.items():
    #    print(f"{key}: {value.shape}")
        
    return shap_dict

# Call the function
shap_dict = reconstruct_shap_values(shap_values, shapes)
sample_to_explain_dict = reconstruct_shap_values(sample_to_explain, shapes)


In [ ]:
shap_abs_sums_dict = {}
for key, value in shap_dict.items():
    shap_abs_sums_dict[key] = np.abs(value).sum()

print(shap_abs_sums_dict)

shap_sums_dict = {}
for k,v in shap_dict.items():
    shap_sums_dict[k] = v.sum()

print(shap_sums_dict)


In [ ]:
len(list(shap_abs_sums_dict.values())), len(list(shap_abs_sums_dict.keys())), sample_to_explain.shape

# Rename keys to be more readable
key_mapping = {
    'meteorological_station_data': 'Meritve meteorološke postaje',
    'past_model_data_tp': 'Padavine prejšnje ALADIN napovedi',
    'future_model_data_tp': 'Padavine trenutne ALADIN napovedi',
    'past_model_data_t': 'Temperatura prejšnje ALADIN napovedi',
    'future_model_data_t': 'Temperatura trenutne ALADIN napovedi',
    'past_model_data_r': 'Relativna vlažnost prejšnje ALADIN napovedi',
    'future_model_data_r': 'Relativna vlažnost trenutne ALADIN napovedi',
    'past_model_data_u10': 'Hitrost vetra-u prejšnje ALADIN napovedi',
    'future_model_data_u10': 'Hitrost vetra-u trenutne ALADIN napovedi',
    'past_model_data_v10': 'Hitrost vetra-v prejšnje ALADIN napovedi',
    'future_model_data_v10': 'Hitrost vetra-v trenutne ALADIN napovedi',
    'past_model_data_diffT500hPa': 'Razlika temperature 500 hPa prejšnje ALADIN napovedi',
    'future_model_data_diffT500hPa': 'Razlika temperature 500 hPa trenutne ALADIN napovedi',
    'river_history': 'Zgodovina pretoka'
}

# Rename keys in shap_abs_sums_dict
shap_abs_sums_dict_renamed = {key_mapping.get(k, k): v for k, v in shap_abs_sums_dict.items()}

# Rename keys in shap_sums_dict
shap_sums_dict_renamed = {key_mapping.get(k, k): v for k, v in shap_sums_dict.items()}

print("Renamed absolute sums:")
print(shap_abs_sums_dict_renamed)
print("\nRenamed sums:")
print(shap_sums_dict_renamed)


In [ ]:
scaler_q = dataset_train.scalers['Q']

# Reverse scale SHAP values with Q scaler
shap_abs_sums_dict_unscaled = {k: scaler_q.inverse_transform([[v]])[0][0] for k, v in shap_abs_sums_dict_renamed.items()}
shap_sums_dict_unscaled = {k: scaler_q.inverse_transform([[v]])[0][0] for k, v in shap_sums_dict_renamed.items()}

shap.bar_plot(
    np.array(list(shap_abs_sums_dict_unscaled.values())), 
    feature_names= list(shap_abs_sums_dict_unscaled.keys()),
    show=False,
    max_display=6
)
plt.title('Feature Importance Based on SHAP Values - absolute')
# Set x-axis limits with rounded up max tick
max_val = max(shap_abs_sums_dict_unscaled.values())
plt.xlim(0, np.ceil(max_val))
plt.tight_layout()
plt.show()

p2ex =shap.Explanation(
        np.array(list(shap_sums_dict_unscaled.values())),
        feature_names=list(shap_sums_dict_unscaled.keys())
    )
shap.plots.bar(
    p2ex,
    show=False,
    max_display=6
)
plt.title('Pomembnost skupin značilk glede na SHAP vrednosti')
plt.xlabel('SHAP vrednost (vpliv na napoved modela)')
# Set x-axis limits with rounded up max tick
max_val = max(shap_sums_dict_unscaled.values())
#plt.xlim(min(shap_sums_dict_unscaled.values()), np.ceil(max_val))
plt.tight_layout()
plt.show()

In [ ]:
print(reconstruct_shap_values(sample_to_explain, shapes)['river_history'].mean())
# Map every row of background data to dict and take mean of river_history
background_data#[:,-18:].mean()

In [ ]:
([shap_dict['meteorological_station_data'], sample_to_explain_dict['meteorological_station_data'].numpy()]),shap_dict['meteorological_station_data']* sample_to_explain_dict['meteorological_station_data'].numpy()

In [ ]:
([shap_dict['river_history'], sample_to_explain_dict['river_history'].numpy()]),shap_dict['river_history']* sample_to_explain_dict['river_history'].numpy()

In [ ]:
scaler_tp = dataset_train.scalers['tp']
scaler_q = dataset_train.scalers['Q']
14.106 - 1

In [ ]:
import hidro_and_meteo_helper_functions as sf

aladin_raw_latitude = np.array([45.202, 45.242, 45.282, 45.322, 45.362, 45.402, 45.442, 45.482, 45.522,
                                        45.562, 45.602, 45.642, 45.682, 45.722, 45.762, 45.802, 45.842, 45.882,
                                        45.922, 45.962, 46.002, 46.042, 46.082, 46.122, 46.162, 46.202, 46.242,
                                        46.282, 46.322, 46.362, 46.402, 46.442, 46.482, 46.522, 46.562, 46.602,
                                        46.642, 46.682, 46.722, 46.762, 46.802, 46.842, 46.882, 46.922, 46.962,
                                        47.002, 47.042])
aladin_raw_longitude = np.array([13.05 , 13.107, 13.164, 13.221, 13.278, 13.335, 13.392, 13.449, 13.506,
                                    13.563, 13.62 , 13.677, 13.734, 13.791, 13.848, 13.905, 13.962, 14.019,
                                    14.076, 14.133, 14.19 , 14.247, 14.304, 14.361, 14.418, 14.475, 14.532,
                                    14.589, 14.646, 14.703, 14.76 , 14.817, 14.874, 14.931, 14.988, 15.045,
                                    15.102, 15.159, 15.216, 15.273, 15.33 , 15.387, 15.444, 15.501, 15.558,
                                    15.615, 15.672, 15.729, 15.786, 15.843, 15.9  , 15.957, 16.014, 16.071,
                                    16.128, 16.185, 16.242, 16.299, 16.356, 16.413, 16.47 , 16.527, 16.584,
                                    16.641, 16.698, 16.755, 16.812, 16.869, 16.926, 16.983, 17.04 ])
print(station_id)
long, lat = sf.get_hidro_station_loc(station_id)
print(long, lat, station_id)
# Find closest longitude and latitude
lon_idx = np.abs(aladin_raw_longitude - long).argmin()
print(aladin_raw_longitude[lon_idx])
lat_idx = np.abs(aladin_raw_latitude - lat).argmin()
print(aladin_raw_latitude[lat_idx])

# Take (5,5) area around it
min_lat = lat_idx - 2
max_lat = lat_idx + 2 + 1
min_lon = lon_idx - 2
max_lon = lon_idx + 2 + 1

x_ticks = aladin_raw_longitude[min_lon:max_lon]
y_ticks = aladin_raw_latitude[min_lat:max_lat]

######################################
######################################






# Station Location Analysis
mydata = shap_dict['future_model_data_tp']
mygt_data = sample_to_explain_dict['future_model_data_tp'].numpy()
print(type(mydata), type(mygt_data))

# Scale ground truth data with scaler_tp
mygt_data_scaled = scaler_tp.inverse_transform(mygt_data.reshape(-1, 1)).reshape(mygt_data.shape)

# Calculate sums
shap_sum = np.sum(mydata, axis=0)
shap_sum = scaler_q.inverse_transform(shap_sum.reshape(-1, 1)).reshape(shap_sum.shape)
print(sum(sum(shap_sum)))
gt_sum = np.sum(mygt_data_scaled, axis=0)

# Create figure with 1x2 subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Get lat/lon values for the selected area
lat_values = aladin_raw_latitude[min_lat:max_lat]
lon_values = aladin_raw_longitude[min_lon:max_lon]
lon_bound_min = (aladin_raw_longitude[min_lon] + aladin_raw_longitude[min_lon-1])/2
lon_bound_max = (aladin_raw_longitude[max_lon-1] + aladin_raw_longitude[max_lon])/2
lat_bound_min = (aladin_raw_latitude[min_lat] + aladin_raw_latitude[min_lat-1])/2
lat_bound_max = (aladin_raw_latitude[max_lat-1] + aladin_raw_latitude[max_lat])/2

# Plot SHAP sums
im1 = ax1.imshow(shap_sum, vmin=shap_sum.min(), vmax=shap_sum.max(), origin='lower',
                 extent=[lon_bound_min, lon_bound_max, lat_bound_min, lat_bound_max])
ax1.set_title('SHAP vrednosti - vsota')
ax1.set_xlabel('Longituda (°)')
ax1.set_ylabel('Latituda (°)')
ax1.set_xticks(lon_values)
ax1.set_yticks(lat_values)
ax1.plot(long, lat, 'ro', markersize=5, label='Hidrološka postaja Žiri')
ax1.legend()
plt.colorbar(im1, ax=ax1)

# Plot ground truth sums
im2 = ax2.imshow(gt_sum, vmin=gt_sum.min(), vmax=gt_sum.max(), origin='lower',
                 extent=[lon_bound_min, lon_bound_max, lat_bound_min, lat_bound_max])
ax2.set_title('Napovedane padavine - vsota (mm)')
ax2.set_xlabel('Zemljepisan dolžina (°)')
ax2.set_ylabel('Zemljepisna širina (°)')
ax2.set_xticks(lon_values)
ax2.set_yticks(lat_values)
ax2.plot(long, lat, 'ro', markersize=5, label='Hidrološka postaja Žiri')
ax2.legend()
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

In [ ]:
mydata = shap_dict['future_model_data_tp']
mydata_shape = mydata.shape

mygt_data = sample_to_explain_dict['future_model_data_tp']

# Get lat/lon values for the selected area
lat_values = aladin_raw_latitude[min_lat:max_lat]
lon_values = aladin_raw_longitude[min_lon:max_lon]

from ipywidgets import interact, IntSlider

def plot_sample(i):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Get global min/max values across all samples
    shap_vmax = (mydata).max()
    shap_vmin = (mydata).min()
    
    gt_vmax = (mygt_data).max()
    
    im1 = ax1.imshow(mydata[i], vmin=shap_vmin, vmax=shap_vmax, origin='lower',
                     extent=[lon_values[0], lon_values[-1], lat_values[0], lat_values[-1]])
    ax1.set_title(f'SHAP Values - Sample {i}')
    ax1.set_xlabel('Longitude')
    ax1.set_ylabel('Latitude')
    ax1.plot(long, lat, 'ro', markersize=5, label='Station Location')
    ax1.legend()
    plt.colorbar(im1, ax=ax1)
    
    im2 = ax2.imshow(mygt_data[i], vmin=0, vmax=gt_vmax, origin='lower',
                     extent=[lon_values[0], lon_values[-1], lat_values[0], lat_values[-1]])
    ax2.set_title(f'Ground Truth - Sample {i}')
    ax2.set_xlabel('Longitude')
    ax2.set_ylabel('Latitude')
    ax2.plot(long, lat, 'ro', markersize=5, label='Station Location')
    ax2.legend()
    plt.colorbar(im2, ax=ax2)
    
    plt.tight_layout()
    plt.show()

interact(plot_sample, i=IntSlider(min=0, max=mydata_shape[0]-1, step=1, description='Sample Index:'))

In [ ]:
mydata = shap_dict['future_model_data_tp']
mydata_shape = mydata.shape

mygt_data = sample_to_explain_dict['future_model_data_tp']

# Get lat/lon values for the selected area
lat_values = aladin_raw_latitude[min_lat:max_lat]
lon_values = aladin_raw_longitude[min_lon:max_lon]

from ipywidgets import interact, IntSlider

def plot_sample(i):
    # Create a single figure with all samples
    n_samples = mydata_shape[0]
    n_cols = 1  # Two columns in each row
    n_rows = (n_samples + 1) // 1  # Calculate number of rows needed
    
    fig = plt.figure(figsize=(20, 5*n_rows))
    
    # Get global min/max values across all samples
    shap_vmax = (mydata).max()
    shap_vmin = (mydata).min()
    gt_vmax = (mygt_data).max()
    
    for idx in range(n_samples):
        # Create subplot for SHAP values
        ax1 = plt.subplot(n_rows, n_cols*2, idx*2 + 1)
        im1 = ax1.imshow(mydata[idx], vmin=shap_vmin, vmax=shap_vmax, origin='lower',
                        extent=[lon_values[0], lon_values[-1], lat_values[0], lat_values[-1]])
        ax1.set_title(f'SHAP Values - Sample {idx}')
        ax1.set_xlabel('Longitude')
        ax1.set_ylabel('Latitude')
        ax1.plot(long, lat, 'ro', markersize=5, label='Station Location')
        ax1.legend()
        plt.colorbar(im1, ax=ax1)
        
        # Create subplot for ground truth
        ax2 = plt.subplot(n_rows, n_cols*2, idx*2 + 2)
        im2 = ax2.imshow(mygt_data[idx], vmin=0, vmax=gt_vmax, origin='lower',
                        extent=[lon_values[0], lon_values[-1], lat_values[0], lat_values[-1]])
        ax2.set_title(f'Ground Truth - Sample {idx}')
        ax2.set_xlabel('Longitude')
        ax2.set_ylabel('Latitude')
        ax2.plot(long, lat, 'ro', markersize=5, label='Station Location')
        ax2.legend()
        plt.colorbar(im2, ax=ax2)
    
    plt.tight_layout()
    plt.show()

# Plot all samples at once
plot_sample(0)

In [ ]:
meteo_station_data = sf.get_k_closest_meteo_stations(station_id, 1)

In [ ]:
shap_values_all = explainer.shap_values(test_data, check_additivity=False)

In [ ]:
shap_values_all.shape

In [ ]:
shap_dict_all = {}
for i in range(len(shap_values_all)):
    shap_dict_all[i] = reconstruct_shap_values(shap_values_all[i].reshape(1, -1), shapes)

shap_abs_sums_dict_all = {}
shap_sums_dict_all = {}



abs_val_row = []
rel_row = []
for i,v in shap_dict_all.items():


    abs_val_row.append([np.abs(value).sum() for k, value in v.items()])
    rel_row.append([value.sum() for k, value in v.items()])

all_abs_rows = np.array(abs_val_row)
all_rel_rows = np.array(rel_row)





shap.plots.beeswarm(
    shap.Explanation(
        values=all_abs_rows,
        feature_names=list(shap_dict_all[0].keys()),
    ),
    color = 'red',
    show=False
)
plt.title('Feature Importance Based on SHAP Values - absolute')
plt.tight_layout()
plt.show()

shap.plots.beeswarm(
    shap.Explanation(
        values=all_rel_rows,
        feature_names=list(shap_dict_all[0].keys()),
    ),
    color = 'blue',
    show=False
)
plt.title('Feature Importance Based on SHAP Values - relative')
plt.tight_layout()



In [ ]:
test_data.shape, all_abs_rows.shape

In [ ]:
# Create a bar plot of SHAP values for each feature
plt.figure(figsize=(15, 8))
features = list(shap_abs_sums_dict.keys())
values = list(shap_abs_sums_dict.values())

# Sort features by their SHAP values
sorted_idx = np.argsort(values)
features = [features[i] for i in sorted_idx]
values = [values[i] for i in sorted_idx]

# Create horizontal bar plot
plt.barh(range(len(features)), values)
plt.yticks(range(len(features)), features)
plt.xlabel('Sum of Absolute SHAP Values')
plt.title('Feature Importance Based on SHAP Values')
plt.grid(True, axis='x', alpha=0.3)

# Add value labels on the bars
for i, v in enumerate(values):
    plt.text(v, i, f'{v:.2f}', va='center')

plt.tight_layout()
plt.show()


In [ ]:
scaled_target.squeeze(1).shape

In [ ]:
np.where(scaled_target.max(axis=2) > np.percentile(scaled_target.max(axis=1), 95))

In [ ]:
scaled_target.max(axis=1).shape

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import numpy as np
from datetime import datetime, timedelta

def plot_predictions(sample_idx):
    plt.figure(figsize=(15, 6))
    print(scaled_output[sample_idx])
    
    # Create x-axis values for 18 hours
    hours = np.arange(18)
    
    # Get the prediction timestamp for this sample and convert to datetime if it's a float
    prediction_time = prediction_timestamps[sample_idx]
    if isinstance(prediction_time, (int, float)):
        prediction_time = datetime.fromtimestamp(prediction_time)
    
    # Create datetime objects for x-axis labels
    time_labels = [prediction_time + timedelta(hours=int(h)) for h in hours]
    
    # Plot predictions and targets
    plt.plot(hours, scaled_output[sample_idx].flatten(), label='Model Prediction', marker='o')
    plt.plot(hours, scaled_target[sample_idx].flatten(), label='Actual Target', marker='x')
    
    # Format the title with date
    plt.title(f'18-hour Prediction for Sample {sample_idx}\nStarting from {prediction_time.strftime("%Y-%m-%d %H:%M")}')
    plt.xlabel('Hours Ahead')
    plt.ylabel('Flow Rate (m³/s)')
    plt.legend()
    plt.grid(True)
    
    # Set x-axis ticks to show dates
    plt.xticks(hours, [t.strftime("%H:%M") for t in time_labels], rotation=45)
    
    # Set y-axis limits based on the maximum value across all scaled targets
    max_value = scaled_target.max()  # Get max across all samples
    min_value = scaled_target.min()  # Get min across all samples
    y_min = min_value * 0.5  # Take 50% of min value
    y_max = max_value * 1.1  # Add 10% padding
    plt.ylim(y_min, y_max)
    
    plt.tight_layout()  # Adjust layout to prevent label cutoff
    plt.show()

# Create an interactive slider
interact(plot_predictions, 
         sample_idx=IntSlider(min=0, 
                            max=len(scaled_output)-1, 
                            step=1, 
                            description='Sample Index:',
                            continuous_update=False))

In [ ]:
# Set model to evaluation mode
model.eval()

# Get a sample batch from test dataset
for batch in dataset_test:  
   
    break

In [ ]:
for key, value in sample_batch.items():
    print(key)
    prin








